In [1]:

import numpy as np
import pandas as pd

from coin_flip_with_riskless_asset_model import CoinFlipWithRisklessAssetModel
from plotting_data import generate_arithmetic_vs_geometric_data
from plotting_functions import create_wealth_plot, create_empirical_growth_rates_plot, create_arithmetic_vs_geometric_plot

from finlib.ensemble_of_returns_paths import EnsembleOfReturnsPaths
from finlib.simulating_performance import crp_performance

import blogkit.brand_plotly as bp   # now resolves

In [2]:
coin_flip_model = CoinFlipWithRisklessAssetModel(gamma_heads=2.0, alpha=0.80, r=0.97, p=0.5)

In [3]:
weights_for_risky_asset = np.array([0.0, 1.0])


In [4]:
opt_result_for_CRP = coin_flip_model.solve_growth_rate_maximization_problem()

f_star = opt_result_for_CRP.x

weights_for_optimal_CRP = np.array([1 - f_star, f_star])

In [5]:
opt_result_for_CRP

 message: Solution found.
 success: True
  status: 0
     fun: -0.012677299157080157
       x: 0.3800034063798751
     nit: 7
    nfev: 7

In [6]:
##### CREATE PERFORMANCE ENSEMBLES FOR THE RISKY ASSET AND THE OPTIMAL CRP STRATEGY #####

seed = 12345
rng = np.random.default_rng(seed)

num_paths = 200
num_periods = 1000

gross_returns_tensor = coin_flip_model.generate_random_gross_returns(rng, num_periods=num_periods, num_paths=num_paths)

perf_risky_matrix = crp_performance(weights_for_risky_asset, gross_returns_tensor)
perf_optimal_CRP_matrix = crp_performance(weights_for_optimal_CRP, gross_returns_tensor)

ensemble_risky = EnsembleOfReturnsPaths.from_gross(perf_risky_matrix)
ensemble_optimal_CRP = EnsembleOfReturnsPaths.from_gross(perf_optimal_CRP_matrix)


In [7]:
##### PLOTTING PREPRARATION #####

# Risky asset summary data
growth_rate_summary_df_risky = ensemble_risky.summarize_across_paths(ensemble_risky.running_growth_rate, threshold=0.0) # threshold is zero because we want to know the fraction of paths with positive growth rate at each period
wealth_summary_df_risky = ensemble_risky.summarize_across_paths(ensemble_risky.running_wealth_ratio, threshold=1.0) # threshold is 1 because we want to know the fraction of paths with a positive net return.

# Optimal CRP summary data
growth_rate_summary_df_optimal_CRP = ensemble_optimal_CRP.summarize_across_paths(ensemble_optimal_CRP.running_growth_rate, threshold=0.0) # threshold is zero because we want to know the fraction of paths with positive growth rate at each period
wealth_summary_df_optimal_CRP = ensemble_optimal_CRP.summarize_across_paths(ensemble_optimal_CRP.running_wealth_ratio, threshold=1.0) # threshold is 1 because we want to know the fraction of paths with a positive net return.

# Arithmetic vs geometric "data"
arith_geo_df = generate_arithmetic_vs_geometric_data(coin_flip_model)


In [8]:
##### WEALTH OVER TIME (spaghetti paths), optimal CRP #####

# create_wealth_plot wants `simulations` as a WIDE frame: one column per path,
# indexed by period. The ensemble's running series come out shaped
# (num_paths, num_periods), so transpose to (num_periods, num_paths) and reuse
# the summary's period index so the spaghetti lines up with the band.
wealth_paths_optimal_CRP = pd.DataFrame(
    ensemble_optimal_CRP.running_wealth_ratio.T,
    index=wealth_summary_df_optimal_CRP.index,
)

create_wealth_plot(
    wealth_paths_optimal_CRP,
    wealth_summary_df_optimal_CRP,
    title="Wealth over time (optimal CRP)",
)

In [9]:
##### EMPIRICAL GROWTH RATE (spaghetti paths), optimal CRP #####

# Same shape fix as above. This plot also takes the CRP weights and the model,
# which it uses to draw and colour the theoretical asymptotic growth rate.
growth_rate_paths_optimal_CRP = pd.DataFrame(
    ensemble_optimal_CRP.running_growth_rate.T,
    index=growth_rate_summary_df_optimal_CRP.index,
)

create_empirical_growth_rates_plot(
    growth_rate_paths_optimal_CRP,
    growth_rate_summary_df_optimal_CRP,
    weights_for_optimal_CRP,
    coin_flip_model,
    title="Empirical growth rate (optimal CRP)",
)

In [ ]:
create_arithmetic_vs_geometric_plot(arith_geo_df, opt_result_for_CRP)